# Applio CEVC — диагностический baseline

Этот блокнот основан на стабильном Applio **3.6.3**.

Что изменено:

- обычная установка и обычный интерфейс Applio сохранены;
- сервер пишет полный технический лог в фоне;
- после запуска появляется кнопка **«Скачать лог»**;
- лог не содержит аудиозаписи и сами модели — только техническую диагностику и список файлов;
- этот запуск нужен для Milestone 0: зафиксировать рабочий baseline до изменения архитектуры.

Порядок:

1. При необходимости подключи Google Drive.
2. Запусти установку.
3. При необходимости синхронизируй модели с Drive.
4. Запусти сервер.
5. Проведи обычную конвертацию.
6. Вернись в последнюю ячейку и нажми **«Скачать лог»**.


In [ ]:
# @title 1. Подключить Google Drive (необязательно)
from google.colab import drive
from google.colab._message import MessageError

try:
    drive.mount("/content/drive")
except MessageError:
    print("❌ Не удалось подключить Google Drive.")


In [ ]:
# @title 2. Установить стабильный Applio и включить диагностику
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import platform
import shutil
import subprocess
import sys

APP_DIR = Path("/content/Applio")
DIAG_DIR = APP_DIR / "cevc_diagnostics"
INSTALL_LOG = DIAG_DIR / "install.log"
ENVIRONMENT_JSON = DIAG_DIR / "environment.json"

REPO_URL = "https://github.com/IAHispano/Applio.git"
REPO_REF = "3.6.3"
NOTEBOOK_BUILD = "cevc-baseline-v1"

def run_logged(command, *, cwd=None, check=True, env=None):
    command = [str(part) for part in command]
    shown = " ".join(command)
    print(f"\n$ {shown}", flush=True)
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    output = completed.stdout or ""
    print(output, end="" if output.endswith("\n") or not output else "\n", flush=True)
    DIAG_DIR.mkdir(parents=True, exist_ok=True)
    with INSTALL_LOG.open("a", encoding="utf-8") as log:
        log.write(f"\n$ {shown}\n")
        log.write(output)
        if output and not output.endswith("\n"):
            log.write("\n")
        log.write(f"[exit_code={completed.returncode}]\n")
    if check and completed.returncode != 0:
        raise subprocess.CalledProcessError(
            completed.returncode, command, output=completed.stdout
        )
    return completed

if not Path("/content").exists():
    raise RuntimeError("Этот блокнот рассчитан на Google Colab.")

if (APP_DIR / ".git").exists():
    print("Найдена существующая установка Applio. Обновляю только код, не удаляя logs.")
    DIAG_DIR.mkdir(parents=True, exist_ok=True)
    run_logged(["git", "fetch", "--depth", "1", "origin", f"refs/tags/{REPO_REF}:refs/tags/{REPO_REF}"], cwd=APP_DIR)
    run_logged(["git", "checkout", "--force", REPO_REF], cwd=APP_DIR)
else:
    if APP_DIR.exists():
        shutil.rmtree(APP_DIR)
    APP_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, "--single-branch", REPO_URL, str(APP_DIR)],
        check=True,
    )
    DIAG_DIR.mkdir(parents=True, exist_ok=True)
    INSTALL_LOG.write_text(
        f"[{datetime.now(timezone.utc).isoformat()}] cloned {REPO_URL} ref {REPO_REF}\n",
        encoding="utf-8",
    )

os.chdir(APP_DIR)

run_logged(["apt-get", "update", "-qq"])
run_logged(["apt-get", "install", "-y", "-qq", "portaudio19-dev", "ffmpeg", "libsndfile1"])
run_logged([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"])
run_logged([
    sys.executable, "-m", "uv", "pip", "install", "--system", "-q",
    "-r", "requirements.txt",
    "--extra-index-url", "https://download.pytorch.org/whl/cu128",
    "--index-strategy", "unsafe-best-match",
])
run_logged([
    sys.executable, "-m", "uv", "pip", "install", "--system", "-q",
    "ngrok", "jupyter-ui-poll", "ipywidgets",
])
run_logged(["npm", "install", "-g", "-q", "localtunnel"])
run_logged([
    sys.executable, "core.py", "prerequisites",
    "--models", "True",
    "--pretraineds_hifigan", "True",
], cwd=APP_DIR)

def command_output(command):
    try:
        result = subprocess.run(
            [str(part) for part in command],
            cwd=APP_DIR,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            timeout=60,
        )
        return {
            "command": " ".join(str(part) for part in command),
            "exit_code": result.returncode,
            "output": result.stdout,
        }
    except Exception as error:
        return {
            "command": " ".join(str(part) for part in command),
            "error": repr(error),
        }

environment = {
    "notebook_build": NOTEBOOK_BUILD,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "python": sys.version,
    "platform": platform.platform(),
    "git_head": command_output(["git", "rev-parse", "HEAD"]),
    "git_status": command_output(["git", "status", "--short"]),
    "nvidia_smi": command_output(["nvidia-smi"]),
}

try:
    import torch
    environment["torch"] = {
        "version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "gpu_count": torch.cuda.device_count(),
        "gpu_names": [
            torch.cuda.get_device_name(index)
            for index in range(torch.cuda.device_count())
        ],
    }
except Exception as error:
    environment["torch_error"] = repr(error)

ENVIRONMENT_JSON.write_text(
    json.dumps(environment, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("✅ Стабильный Applio установлен.")
print(f"Диагностика: {DIAG_DIR}")


In [ ]:
# @title 3. Синхронизировать модели с Google Drive (необязательно)
# @markdown Использует папку `MyDrive/ApplioBackup`, как официальный блокнот.
from pathlib import Path
import os
import shutil

APP_DIR = Path("/content/Applio")
LOCAL_LOGS = APP_DIR / "logs"
DRIVE_ROOT = Path("/content/drive/MyDrive/ApplioBackup")
LOCAL_ONLY = {"mute", "reference", "zips", "mute_spin", "mute_spin-v2"}

if not Path("/content/drive").is_mount():
    print("❌ Google Drive не подключён. Пропускаю синхронизацию.")
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_LOGS.mkdir(parents=True, exist_ok=True)

    local_runtime_root = Path("/tmp/applio_runtime_logs")
    local_runtime_root.mkdir(parents=True, exist_ok=True)

    for name in LOCAL_ONLY:
        current = LOCAL_LOGS / name
        runtime = local_runtime_root / name
        runtime.mkdir(parents=True, exist_ok=True)
        if current.exists() and not current.is_symlink():
            if current.is_dir():
                shutil.copytree(current, runtime, dirs_exist_ok=True)
            else:
                current.unlink()
        if current.is_symlink() or current.exists():
            if current.is_dir() and not current.is_symlink():
                shutil.rmtree(current)
            else:
                current.unlink()
        current.symlink_to(runtime, target_is_directory=True)

    for item in DRIVE_ROOT.iterdir():
        target = LOCAL_LOGS / item.name
        if item.name in LOCAL_ONLY:
            continue
        if target.exists() or target.is_symlink():
            continue
        target.symlink_to(item, target_is_directory=item.is_dir())

    print("✅ Модели из MyDrive/ApplioBackup подключены.")
    print("Новые модели можно сохранять прямо в эту папку на Drive.")


In [ ]:
# @title 4. Запустить Applio с логированием
# @markdown После запуска проведи обычную конвертацию, затем нажми **«Скачать лог»** ниже.
method = "gradio"  # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = ""  # @param {type:"string"}
tensorboard = False  # @param {type:"boolean"}
startup_timeout = 300  # @param {type:"integer"}

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, HTML
from ipywidgets import Button, Output, VBox
import json
import os
import platform
import re
import signal
import subprocess
import sys
import time
import zipfile

from google.colab import files

APP_DIR = Path("/content/Applio")
DIAG_DIR = APP_DIR / "cevc_diagnostics"
APP_LOG = DIAG_DIR / "app.log"
ENVIRONMENT_JSON = DIAG_DIR / "environment.json"
MODEL_MANIFEST = DIAG_DIR / "model_manifest.json"
INFERENCE_SUMMARY = DIAG_DIR / "inference_summary.json"
GPU_SNAPSHOT = DIAG_DIR / "gpu_snapshot.txt"
PIP_FREEZE = DIAG_DIR / "pip_freeze.txt"
PIDS_JSON = DIAG_DIR / "pids.json"

DIAG_DIR.mkdir(parents=True, exist_ok=True)

def process_alive(pid):
    try:
        os.kill(int(pid), 0)
        return True
    except (OSError, ValueError, TypeError):
        return False

def stop_previous():
    if not PIDS_JSON.exists():
        return
    try:
        data = json.loads(PIDS_JSON.read_text(encoding="utf-8"))
    except Exception:
        return
    for pid in data.get("pids", []):
        if not process_alive(pid):
            continue
        try:
            os.killpg(int(pid), signal.SIGTERM)
        except Exception:
            try:
                os.kill(int(pid), signal.SIGTERM)
            except Exception:
                pass
    time.sleep(1)

def append_log(text):
    with APP_LOG.open("a", encoding="utf-8") as log:
        log.write(text)
        if text and not text.endswith("\n"):
            log.write("\n")

def collect_model_manifest():
    records = []
    logs_dir = APP_DIR / "logs"
    if logs_dir.exists():
        for path in logs_dir.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in {".pth", ".onnx", ".index"}:
                continue
            try:
                record = {
                    "relative_path": str(path.relative_to(APP_DIR)),
                    "size_bytes": path.stat().st_size,
                    "modified_at": datetime.fromtimestamp(
                        path.stat().st_mtime, timezone.utc
                    ).isoformat(),
                }
                if path.suffix.lower() == ".pth":
                    try:
                        import torch
                        checkpoint = torch.load(
                            path, map_location="cpu", weights_only=True
                        )
                        weights = (
                            checkpoint.get("weight")
                            or checkpoint.get("model")
                            or checkpoint.get("state_dict")
                        )
                        if isinstance(weights, dict):
                            record["parameter_count"] = int(
                                sum(
                                    tensor.numel()
                                    for tensor in weights.values()
                                    if hasattr(tensor, "numel")
                                )
                            )
                        for key in ("version", "f0", "vocoder", "sr", "speakers_id"):
                            if key in checkpoint:
                                value = checkpoint[key]
                                if isinstance(value, (str, int, float, bool)) or value is None:
                                    record[key] = value
                        config = checkpoint.get("config")
                        if isinstance(config, (list, tuple)) and config:
                            record["config_last_value"] = config[-1]
                        del checkpoint
                    except Exception as error:
                        record["checkpoint_read_error"] = repr(error)
                records.append(record)
            except OSError:
                continue
    MODEL_MANIFEST.write_text(
        json.dumps({"files": records}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def collect_inference_summary():
    text = APP_LOG.read_text(encoding="utf-8", errors="replace") if APP_LOG.exists() else ""
    starts = list(re.finditer(r"Converting audio '(.+?)'\.\.\.", text))
    finishes = list(
        re.finditer(
            r"Conversion completed at '(.+?)' in ([0-9.]+) seconds\.",
            text,
        )
    )
    conversions = []
    finish_index = 0

    for start in starts:
        while finish_index < len(finishes) and finishes[finish_index].start() < start.end():
            finish_index += 1
        if finish_index >= len(finishes):
            conversions.append({
                "input": Path(start.group(1)).name,
                "status": "started_without_completion",
            })
            continue

        finish = finishes[finish_index]
        finish_index += 1
        input_path = Path(start.group(1))
        if not input_path.is_absolute():
            input_path = APP_DIR / input_path
        elapsed = float(finish.group(2))
        record = {
            "input": input_path.name,
            "output": Path(finish.group(1)).name,
            "elapsed_seconds": elapsed,
            "status": "completed",
        }
        try:
            import soundfile as sf
            audio_info = sf.info(str(input_path))
            duration = float(audio_info.frames / audio_info.samplerate)
            record["input_duration_seconds"] = duration
            record["real_time_factor"] = elapsed / duration if duration > 0 else None
            record["input_sample_rate"] = int(audio_info.samplerate)
            record["input_channels"] = int(audio_info.channels)
        except Exception as error:
            record["audio_info_error"] = repr(error)
        conversions.append(record)

    INFERENCE_SUMMARY.write_text(
        json.dumps({"conversions": conversions}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def collect_gpu_snapshot():
    commands = [
        [
            "nvidia-smi",
            "--query-gpu=index,name,driver_version,memory.total,memory.used,utilization.gpu,temperature.gpu",
            "--format=csv",
        ],
        [
            "nvidia-smi",
            "--query-compute-apps=pid,process_name,used_memory",
            "--format=csv",
        ],
    ]
    chunks = []
    for command in commands:
        try:
            result = subprocess.run(
                command,
                text=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                timeout=30,
            )
            chunks.append(f"$ {' '.join(command)}\n{result.stdout}\n")
        except Exception as error:
            chunks.append(f"$ {' '.join(command)}\nERROR: {error!r}\n")
    GPU_SNAPSHOT.write_text("\n".join(chunks), encoding="utf-8")

def refresh_environment():
    data = {}
    if ENVIRONMENT_JSON.exists():
        try:
            data = json.loads(ENVIRONMENT_JSON.read_text(encoding="utf-8"))
        except Exception:
            data = {}
    data.update({
        "log_packaged_at_utc": datetime.now(timezone.utc).isoformat(),
        "server_method": method,
        "python_runtime": sys.version,
        "platform_runtime": platform.platform(),
    })
    try:
        import torch
        data["runtime_torch"] = {
            "version": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_version": torch.version.cuda,
            "gpu_count": torch.cuda.device_count(),
            "gpu_names": [
                torch.cuda.get_device_name(index)
                for index in range(torch.cuda.device_count())
            ],
            "max_memory_allocated_bytes": (
                torch.cuda.max_memory_allocated()
                if torch.cuda.is_available()
                else 0
            ),
            "max_memory_reserved_bytes": (
                torch.cuda.max_memory_reserved()
                if torch.cuda.is_available()
                else 0
            ),
        }
    except Exception as error:
        data["runtime_torch_error"] = repr(error)
    ENVIRONMENT_JSON.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def build_log_bundle():
    collect_model_manifest()
    collect_inference_summary()
    collect_gpu_snapshot()
    refresh_environment()

    freeze = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    PIP_FREEZE.write_text(freeze.stdout or "", encoding="utf-8")

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive = DIAG_DIR / f"CEVC_diagnostic_{stamp}.zip"
    include = [
        DIAG_DIR / "install.log",
        APP_LOG,
        ENVIRONMENT_JSON,
        MODEL_MANIFEST,
        INFERENCE_SUMMARY,
        GPU_SNAPSHOT,
        PIP_FREEZE,
        PIDS_JSON,
    ]
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in include:
            if path.exists() and path.is_file():
                bundle.write(path, arcname=path.name)
    return archive

if method == "ngrok" and not ngrok_token.strip():
    raise ValueError("Для метода ngrok нужен токен.")

stop_previous()
APP_LOG.write_text(
    f"[{datetime.now(timezone.utc).isoformat()}] Starting Applio; method={method}\n",
    encoding="utf-8",
)

environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"
environment["PYTHONIOENCODING"] = "utf-8"

log_handle = APP_LOG.open("a", encoding="utf-8", buffering=1)
pids = []
public_url = None

app_command = [sys.executable, "app.py", "--listen", "--client"]
if method == "gradio":
    app_command.append("--share")

app_process = subprocess.Popen(
    app_command,
    cwd=APP_DIR,
    env=environment,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)
pids.append(app_process.pid)

if tensorboard:
    tensorboard_process = subprocess.Popen(
        [
            sys.executable, "-m", "tensorboard.main",
            "--logdir", str(APP_DIR / "logs"),
            "--bind_all",
            "--port", "6006",
        ],
        cwd=APP_DIR,
        env=environment,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    pids.append(tensorboard_process.pid)

if method == "localtunnel":
    try:
        ip_result = subprocess.run(
            ["curl", "--silent", "https://ipv4.icanhazip.com"],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            timeout=30,
        )
        password_ip = (ip_result.stdout or "").strip()
        if password_ip:
            print(f"Localtunnel Password IP: {password_ip}")
            append_log(f"Localtunnel Password IP: {password_ip}")
    except Exception as error:
        append_log(f"Could not resolve localtunnel password IP: {error!r}")

    tunnel_process = subprocess.Popen(
        ["lt", "--port", "6969"],
        cwd=APP_DIR,
        env=environment,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    pids.append(tunnel_process.pid)

elif method == "ngrok":
    import ngrok
    listener = ngrok.forward(6969, authtoken=ngrok_token.strip())
    public_url = listener.url()
    append_log(f"Ngrok public URL: {public_url}")

PIDS_JSON.write_text(
    json.dumps(
        {
            "started_at_utc": datetime.now(timezone.utc).isoformat(),
            "method": method,
            "pids": pids,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

patterns = [
    re.compile(r"https://[a-zA-Z0-9-]+\.gradio\.live"),
    re.compile(r"https://[^\s]+\.loca\.lt"),
]

printed_lines = 0
deadline = time.time() + max(30, int(startup_timeout))
while time.time() < deadline and public_url is None:
    if app_process.poll() is not None:
        break
    time.sleep(2)
    try:
        text = APP_LOG.read_text(encoding="utf-8", errors="replace")
    except OSError:
        continue

    lines = text.splitlines()
    for line in lines[printed_lines:]:
        print(line)
    printed_lines = len(lines)

    for pattern in patterns:
        match = pattern.search(text)
        if match:
            public_url = match.group(0)
            break

if public_url:
    display(HTML(f'<p><b>Applio:</b> <a href="{public_url}" target="_blank">{public_url}</a></p>'))
else:
    print("⚠️ Публичная ссылка пока не найдена.")
    print(f"Проверь последние строки лога: {APP_LOG}")
    if APP_LOG.exists():
        print("\n".join(APP_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-30:]))

download_output = Output()
download_button = Button(
    description="Скачать лог",
    button_style="primary",
    icon="download",
    tooltip="Собрать и скачать технический лог CEVC",
)

def download_log(_):
    with download_output:
        download_output.clear_output()
        try:
            archive = build_log_bundle()
            print(f"Готово: {archive.name}")
            files.download(str(archive))
        except Exception as error:
            print(f"Не удалось собрать лог: {error!r}")

download_button.on_click(download_log)
display(VBox([download_button, download_output]))

print("✅ Сервер запущен в фоне.")
print("После конвертации нажми «Скачать лог».")
